# Lesson 4.3 — A/B Testing and Iterative Improvement

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Design an A/B test comparing two system variants
- Decide whether a difference is real or noise
- Iterate responsibly: change one variable at a time

**You will produce:** An A/B harness that compares two variants on the eval set with a significance check.

## Concepts

**A/B testing** compares two variants (A = control, B = change) on the same inputs to see which performs better. For LLM systems a 'variant' might be a new prompt, a different model tier, a new chunking strategy, or added re-ranking.

Two disciplines matter: (1) **change one variable at a time** so you can attribute the difference; (2) **check significance** so you do not ship noise — a 2-point accuracy bump on 20 cases may be random. We use a simple proportion comparison; production teams use proper statistical tests and adequate sample sizes. Iteration = hypothesis -> change one thing -> measure on the gold set -> keep or revert. This closes the Phase 1 feedback loop with evidence.

> **Reference:** [Anthropic — Iterating on evaluations](https://docs.anthropic.com/en/docs/test-and-evaluate/develop-tests)

In [ ]:
# !pip install anthropic
import os, json, time, random

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        # Deterministic mock: echo a label when the prompt asks for classification
        return f"[MOCK {model}] {prompt[:60]}"
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Run two variants over the same eval set

We simulate variant results as correctness vectors on the same cases (in practice you run both through your scorer). Same inputs, one changed variable.

In [ ]:
import random
random.seed(7)
n = 60
# Simulated per-case correctness (1=correct). Variant B tweaks the prompt.
A = [1 if random.random() < 0.84 else 0 for _ in range(n)]
B = [1 if random.random() < 0.90 else 0 for _ in range(n)]
accA, accB = sum(A)/n, sum(B)/n
print(f"A accuracy={accA:.3f}  B accuracy={accB:.3f}  delta={accB-accA:+.3f}")

## Code-Along: Step 2 — Check whether the difference is significant

A two-proportion z-test gives a rough signal of whether B truly beats A or it is noise. We implement it with the standard library (no SciPy needed).

In [ ]:
import math
def two_prop_z(a, b):
    na, nb = len(a), len(b)
    pa, pb = sum(a)/na, sum(b)/nb
    p = (sum(a)+sum(b))/(na+nb)
    se = math.sqrt(p*(1-p)*(1/na+1/nb)) or 1e-9
    z = (pb-pa)/se
    return z

z = two_prop_z(A, B)
significant = abs(z) > 1.96  # ~95% confidence
print(f"z={z:.2f} significant@95%={significant}")
print("Decision:", "ship B" if significant and (sum(B)>sum(A)) else "inconclusive - gather more data")

## Exercise

Make it honest: increase `n`, and add a guard that **refuses to declare a winner** when the sample is too small (e.g. n<30 per arm) regardless of z. Then run a scenario where A and B are actually equal (same probability) and confirm your harness usually returns 'inconclusive' — i.e. it does not manufacture false winners.

In [ ]:
# TODO: add min-sample guard to the decision
# TODO: simulate A and B with equal true accuracy and verify mostly 'inconclusive'

## Key Takeaways

- A/B test on identical inputs, changing exactly one variable.
- Check significance before shipping — small samples fool you.
- Iteration is hypothesis -> one change -> measure -> keep/revert.
- A/B evidence is how the feedback loop earns the right to change production.

---

**Next:** [Lesson 4.4 — Diagnosing System Issues](./lesson-4.4-diagnosing-issues.ipynb)